In [1]:
import json, os, sys
from pathlib import Path
from types import SimpleNamespace
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "multiagent" / "paths.py").exists())
os.chdir(ROOT / "multiagent")
sys.path[:0] = [str(ROOT), str(ROOT / "multiagent/images/9_16/exp1")]

import build as exp1
from multiagent.evaluation.utils import load_model, load_tokenizer, eos_ids
from multiagent.model.cache_renderer import load_renderer_checkpoint, flatten_sender_cache, pred_to_cache
from multiagent.model.unaware_comm import full_sender_cache, tuple_to_dynamic_cache
from multiagent.model.vanilla_loader import resolve_model_path
from multiagent.paths import MODELS_ROOT
from multiagent.dataset.c2c import chat_prompt_ids, generation_prompt_ids
from multiagent.dataset.c2c_cross_family import encode_context_pair
from multiagent.dataset.c2c_registry import load_c2c_task

torch.set_grad_enabled(False)

In [2]:
def load_evals(pair, n):
    """Each benchmark's stored SwaT eval: exact task spec, eval args, first n rows."""
    eval_root = ROOT / "multiagent/outputs_mine"
    evals = {}
    for col in exp1.COLUMNS:
        for bench in col.benchmarks:
            if col.setting == "Short":
                path = pair.base(eval_root) / pair.short_dir / f"{bench}_n500_{pair.tag}.json"
            else:
                path = pair.base(eval_root) / pair.directory / f"eval_{bench}_renderer_1x.json"
            d = json.loads(path.read_text())
            evals[bench] = dict(domain=col.domain, setting=col.setting, spec=d["tasks"][0],
                                args=d["evaluation_config"]["arguments"], rows=d["rows"][:n])
            args = evals[bench]["args"]
            print(f"{col.domain:9} {col.setting:7} {bench:24} {evals[bench]['spec']:24} "
                  f"decode_prefix={args['renderer_decode_prefix']:10} max_len={args['max_length']} max_new={args['max_new_tokens']}")
    return evals


def load_pipeline(evals, device, sender_device):
    """SwaT checkpoint used by the evals, plus its sender and receiver."""
    ckpt = next(iter(evals.values()))["args"]["renderer"].removesuffix("/renderer.pt").replace("outputs/", "outputs_mine/", 1)
    swat, payload = load_renderer_checkpoint(Path(ckpt) / "renderer.pt", device)
    sender_path, receiver_path = resolve_model_path(payload["sender_model"]), resolve_model_path(payload["receiver_model"])
    m = SimpleNamespace(
        swat=swat, device=device, sender_device=sender_device,
        position_mode=payload.get("position_mode") or "interp",
        causal=bool(payload["args"].get("causal_cross_attention")),
        sender_tok=load_tokenizer(sender_path), receiver_tok=load_tokenizer(receiver_path),
        sender=load_model(sender_path, torch.bfloat16, sender_device),
        receiver=load_model(receiver_path, torch.bfloat16, device),
    )
    print("SwaT:", ckpt, "\nsender:", sender_path.relative_to(MODELS_ROOT), "\nreceiver:", receiver_path.relative_to(MODELS_ROOT),
          "\nposition_mode:", m.position_mode, "causal cross-attn:", m.causal)
    return m


def load_samples(evals):
    """bench -> list of (sample, stored eval row, grader), same dataset indices the evals used."""
    tasks = {e["spec"]: load_c2c_task(e["spec"]) for e in evals.values()}
    samples = {}
    for bench, e in evals.items():
        task = tasks[e["spec"]]
        samples[bench] = [(task.dataset[r["index"]], r, task.grade) for r in e["rows"]]
        print(f"{bench:24} {len(samples[bench])}")
    return samples


def show_text(title, messages, cut=1500):
    print(f"==== {title} ====")
    if messages is None:
        print("(none: bare assistant header only)\n")
        return
    for msg in messages:
        print(f"[{msg['role']}] {msg['content'][:cut]}{' ...' if len(msg['content']) > cut else ''}\n")


def run_sender(m, sender_text, max_length):
    """Sender prefills sender_text; its cache is de-roped and flattened per token into `memory`.
    encode_context_pair tokenizes sender_text with both tokenizers and records where each receiver token
    sits in the sender sequence (virtual_positions), which SwaT needs."""
    pair_ids = encode_context_pair(m.sender_tok, m.receiver_tok, sender_text, max_length,
                                   m.position_mode, token_alignment=m.swat.token_alignment)
    sender_ids = torch.tensor([pair_ids["sender_ids"]], device=m.sender_device)
    sender_pos = torch.arange(sender_ids.shape[1], device=m.sender_device)[None]
    sender_cache = m.sender(input_ids=sender_ids, position_ids=sender_pos, use_cache=True,
                            past_key_values=full_sender_cache()).past_key_values
    memory = flatten_sender_cache(m.sender, sender_cache, sender_pos).to(m.device)
    print("sender tokens:", sender_ids.shape[1])
    print("sender cache layer 0 K:", tuple(sender_cache.layers[0].keys.shape) if hasattr(sender_cache, "layers") else tuple(sender_cache[0][0].shape))
    print("memory (all layers K+V flattened per token):", tuple(memory.shape))
    return pair_ids, memory


def run_swat(m, pair_ids, memory):
    """Receiver's own embeddings of sender_text cross-attend to the sender memory -> receiver-native cache at positions 0..R-1."""
    receiver_ids = torch.tensor([pair_ids["receiver_ids"]], device=m.device)
    receiver_pos = torch.arange(receiver_ids.shape[1], device=m.device)[None]
    virtual = torch.from_numpy(pair_ids["virtual_positions"][None].astype("float32")).to(m.device)
    limits = torch.from_numpy(pair_ids["cross_causal_limits"][None]).to(m.device) if m.causal else None
    with torch.autocast("cuda", dtype=torch.bfloat16):
        pred = m.swat(m.receiver.get_input_embeddings()(receiver_ids), receiver_pos, memory, virtual, None, limits)
    swatted = pred_to_cache(m.receiver, pred, receiver_pos, torch.bfloat16)
    print("swatter output:", tuple(pred.shape), "= [batch, R, layers, K/V, heads, head_dim]")
    print("swatted cache:", len(swatted), "layers, layer 0 K:", tuple(swatted[0][0].shape))
    return swatted


def run_receiver(m, receiver_text, swatted, max_length, max_new_tokens):
    """Receiver's only inputs: receiver_text (None = bare assistant header) and the swatted cache.
    Prefills its prompt at positions R.. and decodes greedily."""
    if receiver_text is None:
        prompt = generation_prompt_ids(m.receiver_tok)
    else:
        prompt = chat_prompt_ids(m.receiver_tok, receiver_text, max_length, add_generation_prompt=True)
    print("receiver prompt:", repr(m.receiver_tok.decode(prompt)))
    r = swatted[0][0].shape[2]
    cache = tuple_to_dynamic_cache(swatted, m.receiver.config)
    ids = torch.tensor([prompt], device=m.device)
    pos = torch.arange(r, r + len(prompt), device=m.device)[None]
    stop, out = eos_ids(m.receiver_tok), []
    for _ in range(max_new_tokens):
        o = m.receiver(input_ids=ids, position_ids=pos, past_key_values=cache, use_cache=True)
        nxt = int(o.logits[0, -1].argmax())
        if nxt in stop:
            break
        out.append(nxt)
        cache, ids, pos = o.past_key_values, torch.tensor([[nxt]], device=m.device), pos[:, -1:] + 1
    return m.receiver_tok.decode(out, skip_special_tokens=True).strip()


In [3]:
for p in exp1.PAIRS:
    print(f"{p.tag:5} {p.sender:>16} -> {p.receiver}")

PAIR = {p.tag: p for p in exp1.PAIRS}["14E"]
DEVICE = "cuda:2"          # SwaT + receiver
SENDER_DEVICE = "cuda:2"   # sender (15I's Qwen3-32B needs its own GPU)
N = 20                     # samples kept per benchmark (first N rows of the stored eval)

EVALS = load_evals(PAIR, N)
M = load_pipeline(EVALS, DEVICE, SENDER_DEVICE)
SAMPLES = load_samples(EVALS)

14E          Qwen 3 4B -> Llama 3 8B
15A         Llama 3 8B -> Qwen 3 14B
15D     Ministral 3 8B -> Llama 3.2 11B
15E         Gemma 3 4B -> Granite 3.3 8B
15B          Qwen 3 4B -> Gemma 2 9B
15C         Gemma 2 9B -> Ministral 3 14B
15G         Gemma 3 4B -> Llama 3 8B
15H        Gemma 3 12B -> Qwen 3 4B
15I         Qwen 3 32B -> Llama 3 8B
exp1    Granite 3.3 8B -> SmolLM3 3B
In-domain Short   arc-c                    arc-c:test               decode_prefix=generation max_len=2048 max_new=64
In-domain Short   hellaswag                hellaswag:validation     decode_prefix=generation max_len=2048 max_new=64
In-domain Short   winogrande               winogrande:validation    decode_prefix=generation max_len=2048 max_new=64
In-domain Short   mmlu                     mmlu:test                decode_prefix=generation max_len=2048 max_new=64
In-domain Short   gsm8k                    gsm8k:test               decode_prefix=generation max_len=2048 max_new=512
In-domain Context quality_dev    

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

SwaT: outputs_mine/renderer14E_closedbook_seed44_4b_to_llama8b_stage2_dense03/exit-checkpoint-24000 
sender: qwen/Qwen3-4B 
receiver: llama3/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/c4a54320a52ed5f88b7a2f84496903ea4ff07b45 
position_mode: interp causal cross-attn: True


Loading dataset from disk:   0%|          | 0/24 [00:00<?, ?it/s]

arc-c                    20
hellaswag                20
winogrande               20
mmlu                     20
gsm8k                    20
quality_dev              20
qasper_test100           20
hotpotqa_val400          20
musique_val400           20
medqa                    20
openbookqa               20
math-500                 20
boolq_validation         20
race_test                20
longhealth               20
squad_validation20pct    20
narrativeqa_test400      20


In [4]:
BENCH, I = "mmlu", 0
sample, stored, grade = SAMPLES[BENCH][I]
cfg = EVALS[BENCH]["args"]
sender_text = sample.sender_messages
receiver_text = None if cfg["renderer_decode_prefix"] == "generation" else sample.receiver_messages   # closed-book: no text
show_text("SENDER text input", sender_text)
show_text("RECEIVER text input", receiver_text)
print("gold:", sample.target_text)

pair_ids, memory = run_sender(M, sender_text, cfg["max_length"])

==== SENDER text input ====
[user] Answer the following multiple choice question. Respond with only the letter of the correct answer.

Question: This question refers to the following information.
They ask us how we shall govern these new possessions. I answer: Out of local conditions and the necessities of the case methods of government will grow. If England can govern foreign lands, so can America. If Germany can govern foreign lands, so can America. If they can supervise protectorates, so can America. Why is it more difficult to administer Hawaii than New Mexico or California? Both had a savage and an alien population; both were more remote from the seat of government when they came under our dominion than the Philippines are to-day.
Will you say by your vote that American ability to govern has decayed; that a century's experience in self-rule has failed of a result? Will you affirm by your vote that you are an infidel to American power and practical sense? Or will you say that ours 

In [5]:
swatted = run_swat(M, pair_ids, memory)

swatter output: (1, 315, 32, 2, 8, 128) = [batch, R, layers, K/V, heads, head_dim]
swatted cache: 32 layers, layer 0 K: (1, 8, 315, 128)


In [6]:
answer = run_receiver(M, receiver_text, swatted, cfg["max_length"], cfg["max_new_tokens"])

print("answer:", answer)
print("grade :", grade(answer, sample))
print("stored eval output:", stored["output"], "|", stored["grade"])

receiver prompt: '<|start_header_id|>assistant<|end_header_id|>\n\n'
answer: C
grade : correct
stored eval output: C | correct
